In [1]:
# !pip3 install torch
import torch
print("MPS:", torch.backends.mps.is_available())

import sys
print(sys.executable)


MPS: True
/Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/venv_lrl/bin/python


In [ ]:
# api key: echo $MDC_API_KEY
# !pip3 install datasets

from datasets import load_dataset, Features, Value, Audio

features = Features({
    "client_id": Value("string"),
    "path": Value("string"),
    "sentence_id": Value("string"),
    "sentence": Value("string"),
    "sentence_domain": Value("string"),
    "up_votes": Value("string"),
    "down_votes": Value("string"),
    "age": Value("string"),
    "gender": Value("string"),
    "accents": Value("string"),
    "variant": Value("string"),
    "locale": Value("string"),
    "segment": Value("string"),
})

ds = load_dataset(
    "csv",
    data_files={
        "train": "../../data/cv-corpus-23.0-2025-09-05/sw/train.tsv",
        "validation": "../../data/cv-corpus-23.0-2025-09-05/sw/dev.tsv",
        "test": "../../data/cv-corpus-23.0-2025-09-05/sw/test.tsv"
    },
    delimiter="\t",
    features=features,
)


/Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/venv_lrl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 0 examples [00:00, ? examples/s]


DatasetGenerationError: An error occurred while generating the dataset

In [3]:
def fix_path(batch):
    batch["path"] = "/Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/data/cv-corpus-23.0-2025-09-05/sw/clips/" + batch["path"]
    return batch

ds = ds.map(fix_path)
print(ds["train"].features)

{'client_id': Value('string'), 'path': Value('string'), 'sentence_id': Value('string'), 'sentence': Value('string'), 'sentence_domain': Value('string'), 'up_votes': Value('string'), 'down_votes': Value('string'), 'age': Value('string'), 'gender': Value('string'), 'accents': Value('string'), 'variant': Value('string'), 'locale': Value('string'), 'segment': Value('string')}


In [4]:
from datasets import Audio
ds = ds.cast_column("path", Audio(sampling_rate=16000))
ds = ds.remove_columns(['client_id', 'sentence_id', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'locale', 'segment'])
ds = ds.with_format("numpy")
ds = ds.rename_column("path", "audio")

In [5]:
import numpy as np

def inspect_raw_split(split, name="split", n_show=10):
    print(f"\n=== Raw check: {name} ===")
    print("size:", len(split))

    bad = []

    for i, ex in enumerate(split):
        audio = ex["audio"]["array"]
        sr = ex["audio"]["sampling_rate"]
        text = ex["sentence"]

        issues = []

        if audio is None or len(audio) == 0:
            issues.append("empty_audio")

        if not np.isfinite(audio).all():
            if np.isnan(audio).any():
                issues.append("nan_audio")
            if np.isinf(audio).any():
                issues.append("inf_audio")

        if sr != 16000:
            issues.append(f"sampling_rate={sr}")

        if text is None or str(text).strip() == "":
            issues.append("empty_text")

        if issues:
            bad.append({
                "idx": i,
                "issues": issues,
                "text_preview": str(text)[:120]
            })

    print("bad examples:", len(bad))
    for x in bad[:n_show]:
        print(x)

inspect_raw_split(ds["train"], "train")
inspect_raw_split(ds["test"], "test")


=== Raw check: train ===
size: 46611
bad examples: 0

=== Raw check: test ===
size: 11944
bad examples: 0


In [6]:
# !pip3 install transformers

from transformers import WhisperTokenizer, WhisperProcessor, WhisperFeatureExtractor
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", task="transcribe", padding='longest')
feature_extractor = WhisperFeatureExtractor.from_pretrained('openai/whisper-tiny')
processor = WhisperProcessor.from_pretrained('openai/whisper-tiny', task='transcribe')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-tiny')
# model.generation_config.language = "swahili"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

In [8]:
MAX_DURATION = 30.0  
max_label_len = model.config.max_target_positions  
print(max_label_len)

import numpy as np

def prepare_dataset(batch):
    audio = batch["audio"]
    a = audio["array"].astype(np.float32)

    maxabs = np.max(np.abs(a))
    if maxabs > 1.0:
        a = a / maxabs

    batch["input_features"] = feature_extractor(
        a, sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    text = batch["sentence"]
    if isinstance(text, list):
        text = " ".join(text)      
    text = str(text)

    tok = tokenizer(
        text,
        truncation=True,
        max_length=max_label_len, 
    )
    batch["labels"] = tok.input_ids
    batch["label_length"] = len(batch["labels"])

    return batch

448


In [9]:
def is_clean_raw_example(example):
    text = str(example["sentence"])

    if "\t" in text:
        return False
    if "\n" in text:
        return False

    return True

ds = ds.filter(is_clean_raw_example, num_proc=4)

In [10]:
# !pip3 install torchcodec

preprocessed_ds = ds.map(
    prepare_dataset,
    num_proc=4,
    load_from_cache_file=False,
)




Map (num_proc=4): 93218 examples [02:48, 276.72 examples/s]           
Map (num_proc=4): 23380 examples [00:37, 310.21 examples/s]           
Map (num_proc=4): 23882 examples [00:38, 313.93 examples/s]           


In [11]:
preprocessed_ds

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence', 'variant', 'input_features', 'labels', 'label_length'],
        num_rows: 46609
    })
    validation: Dataset({
        features: ['audio', 'sentence', 'variant', 'input_features', 'labels', 'label_length'],
        num_rows: 11690
    })
    test: Dataset({
        features: ['audio', 'sentence', 'variant', 'input_features', 'labels', 'label_length'],
        num_rows: 11941
    })
})

In [12]:
import numpy as np

def inspect_dataset_split(split, name="split"):
    print(f"\n=== Checking {name} ===")
    print("size:", len(split))

    empty_text = 0
    empty_labels = 0
    nan_audio = 0
    inf_audio = 0
    nan_features = 0
    inf_features = 0
    too_long_audio = 0
    too_long_labels = 0

    audio_durations = []
    label_lengths = []

    bad_examples = []

    for i, ex in enumerate(split):
        audio = ex["audio"]["array"]
        sr = ex["audio"]["sampling_rate"]
        text = ex["sentence"]
        feats = ex["input_features"]
        labels = ex["labels"]

        duration = len(audio) / sr
        audio_durations.append(duration)
        label_lengths.append(len(labels))

        issues = []

        if text is None or str(text).strip() == "":
            empty_text += 1
            issues.append("empty_text")

        if len(labels) == 0:
            empty_labels += 1
            issues.append("empty_labels")

        if not np.isfinite(audio).all():
            if np.isnan(audio).any():
                nan_audio += 1
                issues.append("nan_audio")
            if np.isinf(audio).any():
                inf_audio += 1
                issues.append("inf_audio")

        if not np.isfinite(feats).all():
            if np.isnan(feats).any():
                nan_features += 1
                issues.append("nan_features")
            if np.isinf(feats).any():
                inf_features += 1
                issues.append("inf_features")

        if duration > MAX_DURATION:
            too_long_audio += 1
            issues.append(f"audio>{MAX_DURATION}s")

        if len(labels) >= max_label_len:
            too_long_labels += 1
            issues.append(f"labels>={max_label_len}")

        if issues:
            bad_examples.append({
                "idx": i,
                "duration": duration,
                "label_len": len(labels),
                "text_preview": str(text)[:120],
                "issues": issues,
            })

    print("empty_text:", empty_text)
    print("empty_labels:", empty_labels)
    print("nan_audio:", nan_audio)
    print("inf_audio:", inf_audio)
    print("nan_features:", nan_features)
    print("inf_features:", inf_features)
    print(f"audio longer than {MAX_DURATION}s:", too_long_audio)
    print(f"labels >= {max_label_len}:", too_long_labels)

    print("\nDuration stats:")
    print("min:", np.min(audio_durations))
    print("mean:", np.mean(audio_durations))
    print("max:", np.max(audio_durations))

    print("\nLabel length stats:")
    print("min:", np.min(label_lengths))
    print("mean:", np.mean(label_lengths))
    print("max:", np.max(label_lengths))

    print("\nFirst bad examples:")
    for x in bad_examples[:10]:
        print(x)

inspect_dataset_split(preprocessed_ds["train"], "train")
inspect_dataset_split(preprocessed_ds["test"], "test")


=== Checking train ===
size: 46609
empty_text: 0
empty_labels: 0
nan_audio: 0
inf_audio: 0
nan_features: 0
inf_features: 0
audio longer than 30.0s: 0
labels >= 448: 0

Duration stats:
min: 1.188
mean: 5.407354631079835
max: 10.62

Label length stats:
min: 5
mean: 26.49366002274239
max: 72

First bad examples:

=== Checking test ===
size: 11941
empty_text: 0
empty_labels: 0
nan_audio: 0
inf_audio: 0
nan_features: 0
inf_features: 0
audio longer than 30.0s: 0
labels >= 448: 0

Duration stats:
min: 1.296
mean: 5.583373586801775
max: 10.728

Label length stats:
min: 6
mean: 25.016665270915333
max: 70

First bad examples:


In [13]:
for i in [0, 1, 2, 10, 100, 1000, 10297]:
    print("\nIDX:", i)
    print(repr(ds["train"][i]["sentence"]))


IDX: 0
np.str_('deLima alifunga mabao mawili kwenye fainali ya kombe la dunia')

IDX: 1
np.str_('Tembo hawa huishi kwa makundi makundi familia')

IDX: 2
np.str_('wakati utofauti wa maisha duniani kwa ghafla na mkataba mkali')

IDX: 10
np.str_('uchafu wa mavazi yako unasadifu kila kitu juu yako')

IDX: 100
np.str_('Aliamini kwamba kingempa furaha binafsi na pia hamasa ya kukifanya kitu hicho')

IDX: 1000
np.str_('wawekezaji wa dhamana za serikali wanapata faida kubwa')

IDX: 10297
np.str_('Mama anapasi nguo za mjomba')


In [14]:
def is_valid_processed(example):
    if len(example["labels"]) == 0:
        return False
    if len(example["labels"]) >= max_label_len:
        return False
    return True

preprocessed_ds = preprocessed_ds.filter(is_valid_processed, num_proc=4)

In [15]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch
    
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


In [16]:
# !pip3 install evaluate
# !pip3 install jiwer
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

    return {
        "wer": wer,
        "cer": cer,
        "combined": 0.5 * wer + 0.5 * cer,
    }

In [17]:
import math
from transformers import TrainerCallback

class StopOnNaNCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return control

        for key in ["loss", "grad_norm", "eval_loss"]:
            value = logs.get(key, None)
            if value is not None and isinstance(value, (int, float)):
                if math.isnan(value) or math.isinf(value):
                    print(f"\nStopping training because {key} became {value}")
                    control.should_training_stop = True
                    control.should_save = True
                    return control

        return control

In [18]:
# !pip3 install 'accelerate>=1.1.0'

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="../../models/whisper-tiny-sw-no-language-0803",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_steps=700,
    max_steps=7000,
    gradient_checkpointing=True,
    fp16=False,
    eval_strategy="steps",
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    # generation_max_new_tokens=128,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="combined",
    greater_is_better=False,
    push_to_hub=False,
    logging_strategy="steps",
    logging_nan_inf_filter=False,
    max_grad_norm=0.5,
    save_total_limit=2,

)

model.generation_config.max_new_tokens = 128
model.generation_config.num_beams = 1
model.generation_config.do_sample = False


In [19]:
!pip3 install tensorboardX

from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=preprocessed_ds["train"],
    eval_dataset=preprocessed_ds["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[StopOnNaNCallback()],
    # tokenizer=processor.feature_extractor,
)

zsh:1: /Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/venv_lrl/bin/pip3: bad interpreter: /Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/venv/bin/python: no such file or directory
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break

max_steps is given, it will override any value given in num_train_epochs


In [20]:
trainer.train()

  0%|          | 0/7000 [00:00<?, ?it/s]/Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/venv_lrl/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/zuzamakowska/Documents/Africa/Project/Low-resource-languages/venv_lrl/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...
  0%|          | 25/7000

{'loss': 3.8531, 'grad_norm': 48.05975341796875, 'learning_rate': 1.7857142857142858e-07, 'epoch': 0.01}


  1%|          | 50/7000 [01:15<2:51:47,  1.48s/it]

{'loss': 3.7639, 'grad_norm': 41.97746276855469, 'learning_rate': 3.5714285714285716e-07, 'epoch': 0.02}


  1%|          | 75/7000 [01:52<2:50:51,  1.48s/it]

...

 88%|████████▊ | 6150/7000 [6:25:20<21:07,  1.49s/it]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}



Stopping training because loss became nan
{'loss': nan, 'grad_norm': nan, 'learning_rate': 6.746031746031746e-07, 'epoch': 2.11}


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
 88%|████████▊ | 6150/7000 [6:25:21<53:15,  3.76s/it]

{'train_runtime': 23121.2157, 'train_samples_per_second': 4.844, 'train_steps_per_second': 0.303, 'train_loss': nan, 'epoch': 2.11}


TrainOutput(global_step=6150, training_loss=nan, metrics={'train_runtime': 23121.2157, 'train_samples_per_second': 4.844, 'train_steps_per_second': 0.303, 'total_flos': 2.42235058249728e+18, 'train_loss': nan, 'epoch': 2.111044366257616})

In [21]:
# import numpy as np
# np.percentile(preprocessed_ds["validation"]["duration"], [50, 90, 95, 99, 100])

In [22]:
# import numpy as np

# lengths = preprocessed_ds["train"]["label_length"]
# print("min label_length:", min(lengths))
# print("num <= 1:", sum(l <= 1 for l in lengths))
# print("num == 0:", sum(l == 0 for l in lengths))

In [23]:
# print("max label_length:", max(lengths))

In [24]:
# import numpy as np
# import random

# def audio_has_bad_values(ex):
#     a = ex["audio"]["array"]
#     return (not np.isfinite(a).all()) or (np.max(np.abs(a)) > 1.0 + 1e-3)

# idxs = random.sample(range(len(ds["train"])), 1000)
# bad = [i for i in idxs if audio_has_bad_values(ds["train"][i])]
# print("bad in 1000:", len(bad))
# if bad:
#     i = bad[0]
#     a = ds["train"][i]["audio"]["array"]
#     print("example idx:", i, "finite:", np.isfinite(a).all(), "maxabs:", np.max(np.abs(a)))
#     print("sentence:", ds["train"][i]["sentence"][:200])

In [25]:
# import numpy as np
# lengths = np.array(preprocessed_ds["train"]["label_length"])
# print("p95:", np.percentile(lengths, 95))
# print("p99:", np.percentile(lengths, 99))